In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib
from matplotlib.patches import Patch
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter, MultipleLocator


In [ ]:
def extract_tensorboard_data(log_dir, run_name_to_tag, step_start=0, step_end=None):
	# to find out event file
	event_file = None
	for file in os.listdir(log_dir):
		if os.path.isfile(os.path.join(log_dir, file)) and file.startswith('events'):
			event_file = os.path.join(log_dir, file)
			break
	runs_data = {}
	
	# Iterate over each run directory
	for run_name in os.listdir(log_dir):
		# Check if the run is in the list of runs to consider
		if run_name not in run_name_to_tag:
			continue
		if run_name_to_tag[run_name] is None:
			continue
		run_path = os.path.join(log_dir, run_name)
		
		# Load event data from the run directory
		event_acc = EventAccumulator(run_path)
		event_acc.Reload()
		
		# Check if the tag is available in the run
		tag = run_name_to_tag[run_name]
		if tag in event_acc.Tags()['scalars']:
			steps = []
			values = []
			for scalar_event in event_acc.Scalars(tag):
				if scalar_event.step < step_start:
					continue
				if step_end is not None and scalar_event.step > step_end:
					break
				steps.append(scalar_event.step)
				values.append(scalar_event.value)
				
			runs_data[run_name] = {'steps': steps, 'values': values}
		else:
			raise ValueError(f'Tag {tag} not found in run {run_name}')
	
	if event_file is not None:
		event_acc = EventAccumulator(event_file)

		# Load all events from the file
		event_acc.Reload()

		for tag in run_name_to_tag:
			if run_name_to_tag[tag] is None:
				if tag in event_acc.Tags()['scalars']:
					steps = []
					values = []
					for scalar_event in event_acc.Scalars(tag):
						if scalar_event.step < step_start:
							continue
						if step_end is not None and scalar_event.step > step_end:
							break
						steps.append(scalar_event.step)
						values.append(scalar_event.value)
					
					runs_data[tag] = {'steps': steps, 'values': values}

	return runs_data

MODEL_EPISODE_KEY = 'eval_metrics_episode_length_episode_length'
BASELINE_EPISODE_KEY = 'eval_metrics_episode_length_episode_length:dummy'

run_name_to_tag = {
	'eval_metrics_average_speed_all': 'eval_metrics/average_speed',
	'eval_metrics_average_speed_all:dummy': 'eval_metrics/average_speed',
	'eval_metrics_co2_emission_all': 'eval_metrics/co2_emission',
	'eval_metrics_co2_emission_all:dummy': 'eval_metrics/co2_emission',
	'eval_metrics_delay_all': 'eval_metrics/delay',
	'eval_metrics_delay_all:dummy': 'eval_metrics/delay',
	'eval_metrics_flow_all': 'eval_metrics/flow',
	'eval_metrics_flow_all:dummy': 'eval_metrics/flow',
	'eval_metrics_lanechange_count_all': 'eval_metrics/lanechange_count',
	'eval_metrics_lanechange_count_all:dummy': 'eval_metrics/lanechange_count',
	'eval_metrics_total_time_all': 'eval_metrics/total_time',
	'eval_metrics_total_time_all:dummy': 'eval_metrics/total_time',
	'eval_metrics_travel_time_all': 'eval_metrics/travel_time',
	'eval_metrics_travel_time_all:dummy': 'eval_metrics/travel_time',
	'eval_metrics_TTC_5.0_all': 'eval_metrics/TTC_5.0',
	'eval_metrics_TTC_5.0_all:dummy': 'eval_metrics/TTC_5.0',
	'eval_metrics_TET_5.0_all': 'eval_metrics/TET_5.0',
	'eval_metrics_TET_5.0_all:dummy': 'eval_metrics/TET_5.0',
	'eval_metrics_TIT_5.0_all': 'eval_metrics/TIT_5.0',
	'eval_metrics_TIT_5.0_all:dummy': 'eval_metrics/TIT_5.0',
	'action_allow_rate_lane_0_any': 'action_allow_rate/lane_0',
	'action_allow_rate_lane_0_both': 'action_allow_rate/lane_0',
	'action_allow_rate_lane_0_left': 'action_allow_rate/lane_0',
	'action_allow_rate_lane_0_right': 'action_allow_rate/lane_0',
	'action_allow_rate_lane_1_any': 'action_allow_rate/lane_1',
	'action_allow_rate_lane_1_both': 'action_allow_rate/lane_1',
	'action_allow_rate_lane_1_left': 'action_allow_rate/lane_1',
	'action_allow_rate_lane_1_right': 'action_allow_rate/lane_1',
	'action_allow_rate_lane_2_any': 'action_allow_rate/lane_2',
	'action_allow_rate_lane_2_both': 'action_allow_rate/lane_2',
	'action_allow_rate_lane_2_left': 'action_allow_rate/lane_2',
	'action_allow_rate_lane_2_right': 'action_allow_rate/lane_2',
	'action_allow_rate_lane_3_any': 'action_allow_rate/lane_3',
	'action_allow_rate_lane_3_both': 'action_allow_rate/lane_3',
	'action_allow_rate_lane_3_left': 'action_allow_rate/lane_3',
	'action_allow_rate_lane_3_right': 'action_allow_rate/lane_3',
	'action_allow_rate_lane_4_any': 'action_allow_rate/lane_4',
	'action_allow_rate_lane_4_both': 'action_allow_rate/lane_4',
	'action_allow_rate_lane_4_left': 'action_allow_rate/lane_4',
	'action_allow_rate_lane_4_right': 'action_allow_rate/lane_4',
	'eval_metrics_episode_length_episode_length': 'eval_metrics/episode_length',
	'eval_metrics_episode_length_episode_length:dummy': 'eval_metrics/episode_length',
}


In [ ]:
# read tensorboard log data
locations_path = './eval_v3_tf_logs.json'
if locations_path is None:
    raise FileNotFoundError('Cannot find vis_data_locations.json from current working directory')
with open(locations_path, 'r') as f:
    log_dir_locations = json.load(f)
for key, value in log_dir_locations.items():
    globals()[key] = value


In [ ]:
data_dict = {}
data_dict["dummy_015_data"] = extract_tensorboard_data(tf_log_dir_015_dummy_dqn, run_name_to_tag)
data_dict["ld_015_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_dqn, run_name_to_tag)
data_dict["vs_015_data"] = extract_tensorboard_data(tf_log_dir_015_vehicle_stop_dqn, run_name_to_tag)

data_dict["dummy_010_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_dqn, run_name_to_tag)
data_dict["ld_010_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_dqn, run_name_to_tag)
data_dict["vs_010_data"] = extract_tensorboard_data(tf_log_dir_010_vehicle_stop_dqn, run_name_to_tag)

data_dict["dummy_030_data"] = extract_tensorboard_data(tf_log_dir_030_dummy_dqn, run_name_to_tag)
data_dict["ld_030_data"] = extract_tensorboard_data(tf_log_dir_030_lane_degrade_dqn, run_name_to_tag)
data_dict["vs_030_data"] = extract_tensorboard_data(tf_log_dir_030_vehicle_stop_dqn, run_name_to_tag)

# data_dict["dummy_045_data"] = extract_tensorboard_data(tf_log_dir_045_dummy_dqn, run_name_to_tag)
# data_dict["ld_045_data"] = extract_tensorboard_data(tf_log_dir_045_lane_degrade_dqn, run_name_to_tag)
# data_dict["vs_045_data"] = extract_tensorboard_data(tf_log_dir_045_vehicle_stop_dqn, run_name_to_tag)

# data_dict["dummy_010_cr90_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr90_dqn, run_name_to_tag)
# data_dict["dummy_010_cr80_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr80_dqn, run_name_to_tag)
# data_dict["dummy_010_cr70_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr70_dqn, run_name_to_tag)
# data_dict["dummy_010_cr60_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr60_dqn, run_name_to_tag)
# data_dict["dummy_010_cr50_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr50_dqn, run_name_to_tag)

# data_dict["ld_010_cr90_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_cr90_dqn, run_name_to_tag)
# data_dict["ld_010_cr80_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_cr80_dqn, run_name_to_tag)
# data_dict["ld_010_cr70_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_cr70_dqn, run_name_to_tag)
# data_dict["ld_010_cr60_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_cr60_dqn, run_name_to_tag)
# data_dict["ld_010_cr50_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_cr50_dqn, run_name_to_tag)

# data_dict["dummy_010_use_dummy_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_use_dummy_dqn, run_name_to_tag)
# data_dict["ld_010_use_dummy_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_use_dummy_dqn, run_name_to_tag)
# data_dict["vs_010_use_dummy_data"] = extract_tensorboard_data(tf_log_dir_010_vehicle_stop_use_dummy_dqn, run_name_to_tag)
# data_dict["dummy_015_use_dummy_data"] = extract_tensorboard_data(tf_log_dir_015_dummy_use_dummy_dqn, run_name_to_tag)
# data_dict["ld_015_use_dummy_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_use_dummy_dqn, run_name_to_tag)
# data_dict["vs_015_use_dummy_data"] = extract_tensorboard_data(tf_log_dir_015_vehicle_stop_use_dummy_dqn, run_name_to_tag)


In [ ]:
def _resolve_episode_key(metric_key):
	return BASELINE_EPISODE_KEY if metric_key.endswith(':dummy') else MODEL_EPISODE_KEY


def _filter_metric_series(run_data, metric_key, min_epi_length=None, episode_key=None):
	if metric_key not in run_data:
		raise KeyError(f'{metric_key} not found in run data')
	metric_data = run_data[metric_key]
	steps = list(metric_data.get('steps', []))
	values = list(metric_data.get('values', []))
	if min_epi_length is None:
		return steps, values

	episode_key = episode_key or _resolve_episode_key(metric_key)
	episode_data = run_data.get(episode_key)
	if not episode_data:
		return steps, values

	episode_length_by_step = {
		step: value for step, value in zip(episode_data.get('steps', []), episode_data.get('values', []))
	}
	filtered_steps = []
	filtered_values = []
	for step, value in zip(steps, values):
		episode_length = episode_length_by_step.get(step)
		if episode_length is None:
			continue
		if episode_length >= min_epi_length:
			filtered_steps.append(step)
			filtered_values.append(value)
	return filtered_steps, filtered_values


def _filter_metric_pair(run_data, baseline_key, metric_key, min_epi_length=None,
						baseline_episode_key=None, metric_episode_key=None):
	baseline_steps, baseline_values = _filter_metric_series(
		run_data, baseline_key, min_epi_length=min_epi_length, episode_key=baseline_episode_key
	)
	metric_steps, metric_values = _filter_metric_series(
		run_data, metric_key, min_epi_length=min_epi_length, episode_key=metric_episode_key
	)
	baseline_by_step = {step: value for step, value in zip(baseline_steps, baseline_values)}
	metric_by_step = {step: value for step, value in zip(metric_steps, metric_values)}
	common_steps = sorted(set(baseline_by_step) & set(metric_by_step))
	return (
		common_steps,
		[baseline_by_step[step] for step in common_steps],
		[metric_by_step[step] for step in common_steps],
	)


def print_filtered_episode_counts(data_dict, min_epi_length=300,
						model_episode_key=MODEL_EPISODE_KEY,
						baseline_episode_key=BASELINE_EPISODE_KEY):
	print(f'filtered_episode_count (min_epi_length={min_epi_length}) =')
	for run_name, run_data in data_dict.items():
		model_episode_data = run_data.get(model_episode_key)
		baseline_episode_data = run_data.get(baseline_episode_key)
		if not model_episode_data or not baseline_episode_data:
			print(f'  {run_name}: missing episode length series')
			continue
		model_steps = {
			step for step, value in zip(model_episode_data.get('steps', []), model_episode_data.get('values', []))
			if value >= min_epi_length
		}
		baseline_steps = {
			step for step, value in zip(baseline_episode_data.get('steps', []), baseline_episode_data.get('values', []))
			if value >= min_epi_length
		}
		common_steps = model_steps & baseline_steps
		print(
			f'  {run_name}: model={len(model_steps)}/{len(model_episode_data.get("values", []))}, '
			f'dummy={len(baseline_steps)}/{len(baseline_episode_data.get("values", []))}, '
			f'common={len(common_steps)}'
		)


print_filtered_episode_counts(data_dict, min_epi_length=300)


def calculate_all_metrics(data_dict, min_epi_length=None):
	metrics = {'average_speed': ('eval_metrics_average_speed_all', 'eval_metrics_average_speed_all:dummy'),
			   'flow': ('eval_metrics_flow_all', 'eval_metrics_flow_all:dummy'),
			   'co2_emission': ('eval_metrics_co2_emission_all', 'eval_metrics_co2_emission_all:dummy'),
			   'TTC_5.0': ('eval_metrics_TTC_5.0_all', 'eval_metrics_TTC_5.0_all:dummy'),
			   'TET_5.0': ('eval_metrics_TET_5.0_all', 'eval_metrics_TET_5.0_all:dummy')}
	results = {}
	for metric_label, metric_pair in metrics.items():
		if metric_label not in results:
			results[metric_label] = {}
		for data_label, run_data in data_dict.items():
			demand, scenario = data_label.split(':')
			key = (demand, scenario)
			metric_model, metric_baseline = metric_pair
			_, baseline_values, model_values = _filter_metric_pair(
				run_data, metric_baseline, metric_model, min_epi_length=min_epi_length
			)
			if len(model_values) == 0:
				results[metric_label][key] = {'model': (np.nan, np.nan), 'baseline': (np.nan, np.nan)}
				continue
			avg_model = np.mean(model_values)
			std_model = np.std(model_values)
			avg_baseline = np.mean(baseline_values)
			std_baseline = np.std(baseline_values)
			results[metric_label][key] = {'model': (avg_model, std_model), 'baseline': (avg_baseline, std_baseline)}
	return results


def write_metrics_latex_code(data):
	latex_table = r"""\multirow{3}{*}{metric name} & Low  & {baseline_low_stable_flow}  & {model_low_stable_flow} & {baseline_low_lane_degrade}   & {model_low_lane_degrade}  & {baseline_low_vehicle_stop}  & {model_low_vehicle_stop}    \\
									& Medium & {baseline_medium_stable_flow} & {model_medium_stable_flow} & {baseline_medium_lane_degrade} & {model_medium_lane_degrade} & {baseline_medium_vehicle_stop} & {model_medium_vehicle_stop}  \\
									& High & {baseline_high_stable_flow}  & {model_high_stable_flow} & {baseline_high_lane_degrade}  & {model_high_lane_degrade}  & {baseline_high_vehicle_stop}  & {model_high_vehicle_stop}    \\ \hline"""
	metrics = {
		'average_speed': 'Average speed',
		# 'flow': 'Flow',
		'co2_emission': 'CO2 emission',
		'TTC_5.0': 'TTC',
		'TET_5.0': 'TET'
	}
	metrics_norm_round = {
		'average_speed': (1, 2),
		'co2_emission': (1000, 2),
		'TTC_5.0': (100, 2),
		'TET_5.0': (1, 2)
	}
	relative_change_result = {}
	for metric in list(metrics.keys()):
		metric_name = metrics[metric]
		latex_table_new = latex_table.replace("metric name", metric_name)
		relative_change_result[metric] = {}
		for demand in ["low", "medium", "high"]:
			relative_change_result[metric][demand] = {}
			for scenario in ["lane_degrade", "vehicle_stop", "stable_flow"]:
				baseline, model = data[metric][(demand, scenario)]["baseline"], data[metric][(demand, scenario)]["model"]
				norm_factor, n_round = metrics_norm_round[metric]
				baseline_v = f'{round(baseline[0] / norm_factor, n_round)}$\pm${round(baseline[1] / norm_factor, n_round)}'
				model_v = f'{round(model[0] / norm_factor, n_round)}$\pm${round(model[1] / norm_factor, n_round)}'
				relative_change_result[metric][demand][scenario] = round((model[0] - baseline[0]) / baseline[0] * 100, 2)
				latex_table_new = latex_table_new.replace("{baseline_"+demand+"_"+scenario+"}", baseline_v)
				latex_table_new = latex_table_new.replace("{model_"+demand+"_"+scenario+"}", model_v)
		print(latex_table_new)
	def _to_python_float_dict(value):
		if isinstance(value, dict):
			return {k: _to_python_float_dict(v) for k, v in value.items()}
		return float(value)

	print('relative_change_result =')
	print(_to_python_float_dict(relative_change_result))
	print('average_speed_uplift =')
	print(_to_python_float_dict(relative_change_result['average_speed']))
	return relative_change_result


In [ ]:
metrics_data_dict = {
	'medium:stable_flow': data_dict["dummy_015_data"],
	'medium:lane_degrade': data_dict["ld_015_data"],
	'medium:vehicle_stop': data_dict["vs_015_data"],
	'low:stable_flow': data_dict["dummy_010_data"],
	'low:lane_degrade': data_dict["ld_010_data"],
	'low:vehicle_stop': data_dict["vs_010_data"],
	'high:stable_flow': data_dict["dummy_030_data"],
	'high:lane_degrade': data_dict["ld_030_data"],
	'high:vehicle_stop': data_dict["vs_030_data"]
}

results = calculate_all_metrics(metrics_data_dict, min_epi_length=300)
relative_change_result = write_metrics_latex_code(results)


In [ ]:
# Alternative comparison when use-dummy evaluation logs are available:
# metrics_data_dict = {
# 	'medium:stable_flow': data_dict["dummy_015_data"],
# 	'medium:lane_degrade': data_dict["ld_015_use_dummy_data"],
# 	'medium:vehicle_stop': data_dict["vs_015_use_dummy_data"],
# 	'low:stable_flow': data_dict["dummy_010_data"],
# 	'low:lane_degrade': data_dict["ld_010_use_dummy_data"],
# 	'low:vehicle_stop': data_dict["vs_010_use_dummy_data"],
# 	'high:stable_flow': data_dict["dummy_030_data"],
# 	'high:lane_degrade': data_dict["ld_030_data"],
# 	'high:vehicle_stop': data_dict["vs_030_data"]
# }
#
# results = calculate_all_metrics(metrics_data_dict, min_epi_length=300)
# write_metrics_latex_code(results)


In [ ]:
def plot_control_rate_performance(data_dict, save_pth=None, kargs={}):
	metrics = {'average speed': ('eval_metrics_average_speed_all', 'eval_metrics_average_speed_all:dummy'),
			   'CO2 emission': ('eval_metrics_co2_emission_all', 'eval_metrics_co2_emission_all:dummy')
	}
	metrics_labels = list(metrics.keys())
	results = {}
	min_epi_length = kargs.get('min_epi_length')
	for metric_label, metric_pair in metrics.items():
		if metric_label not in results:
			results[metric_label] = {}
		for data_label, run_data in data_dict.items():
			metric_model, metric_baseline = metric_pair
			_, baseline_values, model_values = _filter_metric_pair(
				run_data, metric_baseline, metric_model, min_epi_length=min_epi_length
			)
			if len(model_values) == 0:
				uplift = np.nan
			else:
				avg_model = np.mean(model_values)
				avg_baseline = np.mean(baseline_values)
				uplift = 0. if avg_baseline == 0. else round((avg_model - avg_baseline) / avg_baseline * 100, 2)
			results[metric_label][data_label] = uplift
	font = {
		'family' : 'Times New Roman',
		'size': 8
		}
	if 'font' in kargs:
		font.update(kargs['font'])
	matplotlib.rc('font', **font)
	figsize = kargs.get('figsize', (3, 2))
	fig, ax = plt.subplots(figsize=figsize)
	colors = ['r', 'b']
	makers = ['o', '^']
	for idx, metric in enumerate(metrics_labels):
		x = list(data_dict.keys())
		y = [results[metric][key] for key in x]
		ax.plot(x, y, label=metric, color=colors[idx], marker=makers[idx], markerfacecolor='white')
	ax.set_xlabel('Regulation rate (%)')
	ax.set_ylabel(kargs.get('ylabel', 'Uplift (%)'))
	plt.grid(axis='y', linestyle='dotted', linewidth=1)
	if kargs.get('ylim'):
		plt.ylim(kargs['ylim'])
	if save_pth:
		plt.savefig(save_pth, dpi=300, bbox_inches='tight')
	plt.show()

# Example when control-rate evaluation logs are available:
# data_dict_015_ld = {
# 	'100': data_dict["ld_015_data"],
# 	'90': data_dict["ld_010_cr90_data"],
# 	'80': data_dict["ld_010_cr80_data"],
# 	'70': data_dict["ld_010_cr70_data"],
# 	'60': data_dict["ld_010_cr60_data"],
# 	'50': data_dict["ld_010_cr50_data"]
# }
#
# data_dict_010_dummy = {
# 	'100': data_dict["dummy_010_data"],
# 	'90': data_dict["dummy_010_cr90_data"],
# 	'80': data_dict["dummy_010_cr80_data"],
# 	'70': data_dict["dummy_010_cr70_data"],
# 	'60': data_dict["dummy_010_cr60_data"],
# 	'50': data_dict["dummy_010_cr50_data"]
# }


In [ ]:
def plot_data_mean_and_stdv(datas, keys, labels, colors, title=None, save_pth=None, window_size=10, kargs={}):
	save_folder = os.path.dirname(save_pth) if save_pth else None
	if save_folder and not os.path.exists(save_folder):
		os.makedirs(save_folder)
	plt.subplots(figsize=(6, 6))
	x_values = np.arange(1, len(datas)+1)
	min_epi_length = kargs.get('min_epi_length')
	for key, label, color in zip(keys, labels, colors):
		means, stds = [], []
		for data in datas:
			_, values = _filter_metric_series(data, key, min_epi_length=min_epi_length)
			means.append(np.mean(values))
			stds.append(np.std(values))
		plt.plot(x_values, means, marker='o', linestyle='-', color=color, label=label)
		plt.fill_between(x_values, [m - s for m, s in zip(means, stds)],
						[m + s for m, s in zip(means, stds)], color=color, alpha=kargs.get('alpha', 0.2))

	plt.xticks(kargs.get('xticks', x_values))
	plt.xlabel(kargs.get('xlabel', 'Learning steps'))
	plt.ylabel(kargs.get('ylabel', 'Average speed (m/s)'))
	if title:
		plt.title(title)
	if kargs.get('xlim'):
		plt.xlim(kargs['xlim'])
	if kargs.get('ylim'):
		plt.ylim(kargs['ylim'])
	plt.grid(linestyle='dotted', linewidth=1)
	plt.legend()
	if save_pth:
		plt.savefig(save_pth)
	plt.show()


def _resolve_boxplot_series(series_spec, min_epi_length=None):
	if isinstance(series_spec, dict) and 'steps' in series_spec and 'values' in series_spec:
		return list(series_spec['values'])
	if isinstance(series_spec, (tuple, list)):
		if len(series_spec) == 2:
			run_data, metric_key = series_spec
			_, values = _filter_metric_series(run_data, metric_key, min_epi_length=min_epi_length)
			return values
		if len(series_spec) == 3:
			run_data, metric_key, episode_key = series_spec
			_, values = _filter_metric_series(run_data, metric_key, min_epi_length=min_epi_length, episode_key=episode_key)
			return values
	raise ValueError('Unsupported series spec for boxplot')


def plot_data_boxplot_with_baseline(data_dict, colors, title=None, save_pth=None, kargs={}):
	save_folder = os.path.dirname(save_pth) if save_pth else None
	if save_folder and not os.path.exists(save_folder):
		os.makedirs(save_folder)
	matplotlib.rcParams['axes.linewidth'] = 0.5
	font = {
		'family' : 'Times New Roman',
		'size': 8
		}
	if 'font' in kargs:
		font.update(kargs['font'])
	matplotlib.rc('font', **font)
	plt.subplots(figsize=(4, 4))
	labels = list(data_dict.keys())
	min_epi_length = kargs.get('min_epi_length')
	box_data = [_resolve_boxplot_series(data_dict[label], min_epi_length=min_epi_length) for label in labels]
	box_width = kargs.get('box_width', 0.5)
	flierprops = {'marker':'D', 'markerfacecolor':'black', 'markersize':2, 'markeredgecolor':'black'}
	bplot = plt.boxplot(box_data, patch_artist=True, labels=labels, widths=box_width, 					 flierprops=flierprops, showfliers=kargs.get('showfliers', True))
	for patch, color in zip(bplot['boxes'], colors):
		patch.set_facecolor(color)
	legend_elements = [Patch(facecolor=color, label=label) for color, label in zip(colors, labels)]
	if kargs.get('legend', True):
		bbox_pos = kargs.get('bbox_pos', (0.99, 0.99))
		plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=bbox_pos, fontsize=6)
	plt.xlabel(kargs.get('xlabel', 'Learning steps'))
	plt.ylabel(kargs.get('ylabel', 'Average speed (m/s)'))
	plt.grid(axis='y', linestyle='dotted', linewidth=1)
	if title:
		plt.title(title)
	if kargs.get('xlim'):
		plt.xlim(kargs['xlim'])
	if kargs.get('ylim'):
		plt.ylim(kargs['ylim'])
	if save_pth:
		plt.savefig(save_pth, dpi=300, bbox_inches='tight')
	plt.show()


def plot_data_uplift_boxplot(data_dict, colors, title=None, save_pth=None, kargs={}):
	save_folder = os.path.dirname(save_pth) if save_pth else None
	if save_folder and not os.path.exists(save_folder):
		os.makedirs(save_folder)
	matplotlib.rcParams['axes.linewidth'] = 0.5
	font = {
		'family' : 'Times New Roman',
		'size': 8
		}
	if 'font' in kargs:
		font.update(kargs['font'])
	matplotlib.rc('font', **font)
	figsize = kargs.get('figsize', (4, 4))
	fig, ax = plt.subplots(figsize=figsize)
	labels = list(data_dict.keys())
	min_epi_length = kargs.get('min_epi_length')
	box_data = []
	for label in labels:
		series_spec = data_dict[label]
		if isinstance(series_spec, (tuple, list)) and len(series_spec) >= 3 and isinstance(series_spec[0], dict):
			run_data, baseline_key, metric_key = series_spec[:3]
			baseline_episode_key = series_spec[3] if len(series_spec) > 3 else None
			metric_episode_key = series_spec[4] if len(series_spec) > 4 else None
			_, baseline_values, metric_values = _filter_metric_pair(
				run_data,
				baseline_key,
				metric_key,
				min_epi_length=min_epi_length,
				baseline_episode_key=baseline_episode_key,
				metric_episode_key=metric_episode_key,
			)
		else:
			v1, v2 = series_spec[0]['values'], series_spec[1]['values']
			limit = min(len(v1), len(v2))
			baseline_values, metric_values = v1[:limit], v2[:limit]
		if len(metric_values) == 0:
			box_data.append([])
			continue
		uplift = [(metric_values[i] - baseline_values[i]) / baseline_values[i] * 100 for i in range(len(metric_values)) if baseline_values[i] != 0]
		box_data.append(uplift)
	box_width = kargs.get('box_width', 0.5)
	flierprops = {'marker':'D', 'markerfacecolor':'black', 'markersize':2, 'markeredgecolor':'black'}
	bplot = plt.boxplot(box_data, patch_artist=True, labels=labels, widths=box_width, 					 flierprops=flierprops, showfliers=kargs.get('showfliers', True))
	for patch, color in zip(bplot['boxes'], colors):
		patch.set_facecolor(color)
	legend_elements = [Patch(facecolor=color, label=label) for color, label in zip(colors, labels)]
	if kargs.get('legend', True):
		bbox_pos = kargs.get('bbox_pos', (0.99, 0.99))
		legend_fontsize = kargs.get('legend_fontsize', 6)
		plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=bbox_pos, fontsize=legend_fontsize)
	plt.xlabel(kargs.get('xlabel', 'Learning steps'))
	plt.ylabel(kargs.get('ylabel', 'Average speed (m/s)'))
	plt.grid(axis='y', linestyle='dotted', linewidth=1)
	if title:
		plt.title(title)
	if kargs.get('xlim'):
		plt.xlim(kargs['xlim'])
	if kargs.get('ylim'):
		plt.ylim(kargs['ylim'])
	if kargs.get('y_space'):
		ax.yaxis.set_major_locator(ticker.MultipleLocator(kargs.get('y_space')))
	if save_pth:
		plt.savefig(save_pth, dpi=300, bbox_inches='tight')
	plt.show()


def plot_action_histogram(run_data, colors, title=None, save_pth=None, kargs={}):
	save_folder = os.path.dirname(save_pth) if save_pth else None
	if save_folder and not os.path.exists(save_folder):
		os.makedirs(save_folder)
	actions = ['any', 'both', 'left', 'right']
	labels = ['Any', 'Left & Right', 'Left only', 'Right only']
	lanes = [0, 1, 2, 3, 4]
	min_epi_length = kargs.get('min_epi_length')
	averages = {}
	for lane in lanes:
		lane_values = []
		for action in actions:
			metric_key = f'action_allow_rate_lane_{lane}_{action}'
			_, values = _filter_metric_series(run_data, metric_key, min_epi_length=min_epi_length)
			lane_values.append(np.mean(values) if len(values) > 0 else np.nan)
		averages[lane] = lane_values
	font = {
		'family' : 'Times New Roman',
		'size': 8
		}
	if 'font' in kargs:
		font.update(kargs['font'])
	matplotlib.rc('font', **font)
	matplotlib.rcParams['axes.linewidth'] = 0.5
	figsize = kargs.get('figsize', (5, 2.5))
	fig, ax = plt.subplots(figsize=figsize)
	width = 0.15
	x = np.arange(len(lanes))
	bar_gap = 0.02
	for i, action in enumerate(actions):
		offsets = x - 1.5 * width + i * (width + bar_gap)
		ax.bar(offsets, [averages[lane][i] for lane in lanes], width, label=labels[i].capitalize(), color=colors[i])
	ax.set_ylabel('Action Rate')
	if title:
		ax.set_title(title)
	ax.set_xticks(x + 0.2 * width)
	ax.set_xticklabels([f'Lane {i+1}' for i in lanes])
	ax.set_ylim(0, 1)
	legend_fontsize = kargs.get('legend_fontsize', 6)
	ax.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=False, framealpha=0.0, fontsize=legend_fontsize)
	plt.grid(axis='y', linestyle='dotted', linewidth=1)
	if save_pth:
		plt.savefig(save_pth, dpi=300, bbox_inches='tight')
	plt.show()


In [ ]:
key = 'eval_metrics_average_speed_all'
key_dummy = 'eval_metrics_average_speed_all:dummy'
uplift_data_dict = {
	'stable flow ': (data_dict["dummy_015_data"], key_dummy, key),
	'lane degrade': (data_dict["ld_015_data"], key_dummy, key),
	'vehicle stop': (data_dict["vs_015_data"], key_dummy, key)
}
colors = ['blue', 'green', 'red']
save_pth = './eval_vis_new/015_uplift_boxplot.pdf'
kargs = {'xlabel': '', 'ylabel': 'Average speed uplift(%)', 'legend': False,
		 'box_width': 0.6, 'ylim': (-5, 10), 'figsize': (2.5, 2.5),
		 'font': {'family' : 'Times New Roman','size': 9}, 'min_epi_length': 300}
plot_data_uplift_boxplot(uplift_data_dict, colors, save_pth=save_pth, kargs=kargs)


key = 'eval_metrics_average_speed_all'
key_dummy = 'eval_metrics_average_speed_all:dummy'
uplift_data_dict = {
	'stable flow ': (data_dict["dummy_010_data"], key_dummy, key),
	'lane degrade': (data_dict["ld_010_data"], key_dummy, key),
	'vehicle stop': (data_dict["vs_010_data"], key_dummy, key)
}
colors = ['blue', 'green', 'red']
save_pth = './eval_vis_new/010_uplift_boxplot.pdf'
kargs = {'xlabel': '', 'ylabel': 'Average speed uplift(%)', 'legend': False,
		 'box_width': 0.6, 'ylim': (-5, 10), 'figsize': (2.5, 2.5),
		 'font': {'family' : 'Times New Roman','size': 9}, 'min_epi_length': 300}
plot_data_uplift_boxplot(uplift_data_dict, colors, save_pth=save_pth, kargs=kargs)


key = 'eval_metrics_average_speed_all'
key_dummy = 'eval_metrics_average_speed_all:dummy'
uplift_data_dict = {
	'stable flow ': (data_dict["dummy_030_data"], key_dummy, key),
	'lane degrade': (data_dict["ld_030_data"], key_dummy, key),
	'vehicle stop': (data_dict["vs_030_data"], key_dummy, key)
}
colors = ['blue', 'green', 'red']
save_pth = './eval_vis_new/030_uplift_boxplot.pdf'
kargs = {'xlabel': '', 'ylabel': 'Average speed uplift(%)', 'legend': False,
		 'box_width': 0.6, 'ylim': (-5, 10), 'figsize': (2.5, 2.5),
		 'font': {'family' : 'Times New Roman','size': 9}, 'min_epi_length': 300}
plot_data_uplift_boxplot(uplift_data_dict, colors, save_pth=save_pth, kargs=kargs)


In [ ]:
colors = ['blue', 'green', 'red', 'skyblue']
save_pth = './eval_vis_new/stable_flow_015_action_allow_rate.pdf'
kargs = {'figsize': (3, 2), 'font': {'family' : 'Times New Roman','size': 10},
		 'legend_fontsize': 8, 'min_epi_length': 300}
plot_action_histogram(data_dict['dummy_015_data'], colors, title=None, save_pth=save_pth, kargs=kargs)


In [ ]:
colors = ['blue', 'green', 'red', 'skyblue']
save_pth = './eval_vis_new/stable_flow_010_action_allow_rate.pdf'
kargs = {'figsize': (3, 2), 'font': {'family' : 'Times New Roman','size': 10},
		 'legend_fontsize': 8, 'min_epi_length': 300}
plot_action_histogram(data_dict['dummy_010_data'], colors, title=None, save_pth=save_pth, kargs=kargs)
